### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [180]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [181]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [182]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [183]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [184]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [185]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [186]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [187]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [188]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palabra que no está en el documento.

In [189]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [190]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [191]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [192]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [193]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [194]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [195]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [196]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750])

Obtenemos los 5 documentos más similares:

In [197]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [198]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [199]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [200]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [201]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [202]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


### Desafío 1.1: Similaridad entre documentos

Se seleccionan 5 documentos aleatorios del conjunto de test y se buscan los 5 documentos más similares a cada uno (también del conjunto de test). Se analiza si la similaridad tiene sentido según el contenido y las etiquetas.

In [203]:
# Selección de documentos aleatorios (con seed para reproducibilidad)
np.random.seed(17)
sample_indices = np.random.choice(len(newsgroups_test.data), size=5, replace=False)
print(f"Índices seleccionados: {sample_indices}")

Índices seleccionados: [1523 5467 1067 5637 4895]


In [204]:
def get_top_n_similar(doc_idx, X, n=5, exclude_self=True):
    """
    Encuentra los n documentos más similares a un documento dado.
    
    Returns: lista de tuplas (índice, score)
    """
    similarities = cosine_similarity(X[doc_idx], X)[0]
    if exclude_self:
        similarities[doc_idx] = -1
    
    top_n = np.argpartition(similarities, -n)[-n:]
    top_n_sorted = top_n[np.argsort(similarities[top_n])[::-1]]
    
    return [(idx, similarities[idx]) for idx in top_n_sorted]


def get_label_name(label_id):
    """Obtiene el nombre de la categoría dado su ID."""
    return newsgroups_test.target_names[label_id]

In [205]:
def analyze_document_similarity(doc_idx, X, data, targets, n_similar=5):
    """
    Analiza un documento y sus n más similares, mostrando un resumen estructurado.
    """
    doc_label = get_label_name(targets[doc_idx])
    
    print(f"{'='*80}")
    print(f"DOCUMENTO ORIGINAL (idx={doc_idx}) | Categoría: {doc_label}")
    print(f"{'='*80}")
    print(data[doc_idx])
    print()
    
    similar_docs = get_top_n_similar(doc_idx, X, n=n_similar)
    
    print(f"{'─'*80}")
    print(f"TOP {n_similar} DOCUMENTOS MÁS SIMILARES:")
    print(f"{'─'*80}")
    
    match_count = 0
    for rank, (sim_idx, score) in enumerate(similar_docs, 1):
        sim_label = get_label_name(targets[sim_idx])
        match = "✓" if sim_label == doc_label else "✗"
        if sim_label == doc_label:
            match_count += 1
        
        print(f"\n{rank}. (idx={sim_idx}) [Score: {score:.3f}] [{match}] Categoría: {sim_label}")
        print(f"{'─'*40}")
        print(data[sim_idx])
        print()
    
    print(f"→ Coincidencia de categoría: {match_count}/{n_similar} ({match_count/n_similar*100:.0f}%)")
    print("\n")

In [206]:
analyze_document_similarity(sample_indices[0], X_test, newsgroups_test.data, y_test)

DOCUMENTO ORIGINAL (idx=1523) | Categoría: rec.sport.baseball


Have there ever been any other no-hitters in Mariner history?

────────────────────────────────────────────────────────────────────────────────
TOP 5 DOCUMENTOS MÁS SIMILARES:
────────────────────────────────────────────────────────────────────────────────

1. (idx=5938) [Score: 0.543] [✓] Categoría: rec.sport.baseball
────────────────────────────────────────
} 
} >Oh, yea, and Chris Bosio pitched a NO-HITTER.  One over the minimum, two
} 
} Have there ever been any other no-hitters in Mariner history?

Randy Johnson, June 2, 1990 against the Tigers.


2. (idx=5777) [Score: 0.149] [✓] Categoría: rec.sport.baseball
────────────────────────────────────────

And not the only quality Mariner pitcher.  I logged on expecting to see
at least ONE congratulatory note for Chris Bosio's NO HITTER, but nary
a peep.  

So I'll take this opportunity to note that the red feet are now 11-5 and
slinking out of town without having scored a 

#### Análisis documento 1 (idx=1523)

El texto 1 es muy corto, consiste en solo una frase cuya categoría es Baseball. La frase pregunta si ha existido un jugador "no-hitter" en la historia de los Mariner. El término "no-hitter" hace referencia a lanzadores cuyas pelotas no han sido bateadas durante todo un partido.

**Documento similar 1:** Aparentemente es la respuesta a la pregunta del texto original dado que responde que el jugador Chris Bosio, por lo menos, logró un "no-hitter". La similitud es alta pues contiene todo el texto del documento original y varios términos se repiten.

**Documento similar 2:** También la categoría es Baseball. Se repiten términos como Mariner, no-hitter, pitcher. Si bien el score es bastante más bajo, la relación es clara: el texto habla de baseball, habla de no-hitters en el mismo equipo de los Mariners, aunque es bastante más largo y agrega información sobre otros equipos.

**Documento similar 3:** Al igual que los anteriores, es de la categoría correspondiente. Se repiten los términos pitcher y hitter, aunque el principal tema del documento son los bateadores y no los lanzadores.

**Documento similar 4:** En este caso solo se repite el término no-hitter. La relación con el texto original es baja aunque hable de la misma temática.

**Documento similar 5:** Este documento tiene muy poca relación con el original, la temática es completamente diferente. La única similitud encontrada es sobre la palabra "history": en el documento original se preguntaba sobre la historia del equipo pero en este documento se habla de la historia del pueblo armenio.

**Conclusión:** Este caso es muy interesante ya que con muy pocas palabras se encontraron relaciones muy relevantes. Los primeros cuatro casos son del mismo tema; si bien el texto era corto, era muy específico del tema. Sin embargo, con tan poca información no se encontraron más coincidencias con textos de la misma temática y el último resultado está realmente alejado de la temática original.

In [207]:
analyze_document_similarity(sample_indices[1], X_test, newsgroups_test.data, y_test)

DOCUMENTO ORIGINAL (idx=5467) | Categoría: sci.electronics
I have seen the existance of electronics solder with a 2% silver
content that seems to have good wetting and fatique reatings.
  Can anyone tell me why it is not used? (silver is not such an expensive
metal).


Andy

────────────────────────────────────────────────────────────────────────────────
TOP 5 DOCUMENTOS MÁS SIMILARES:
────────────────────────────────────────────────────────────────────────────────

1. (idx=5611) [Score: 0.683] [✓] Categoría: sci.electronics
────────────────────────────────────────
:   I have seen the existance of electronics solder with a 2% silver
: content that seems to have good wetting and fatique reatings.
:   Can anyone tell me why it is not used? (silver is not such an expensive
: metal).
: 
: 
: Andy
: 

For the most part, silver-solder is not used for general soldering
tasks due to the mechanism of dendritic growth. Silver-solder, when
exposed to high humidity and placed in an electric field,

#### Análisis documento 2 (idx=5467)

El texto original es una entrada en un post. Explica la experiencia de un soldador al realizar una soldadura con 2% de plata y pregunta por qué su uso no es tan frecuente si la plata no es un metal tan caro.

**Documento similar 1:** Al igual que en las similitudes del documento 1, el documento de mayor similitud en este caso es una respuesta a la pregunta del texto original donde está citado el texto completo. Todas las palabras se repiten y, dado que es una respuesta, tiene coherencia que sea el texto más similar.

**Documento similar 2:** Este documento trata el mismo tema: la soldadura con plata. Incluso menciona el uso de 2% del metal. Varias palabras clave se repiten, aunque agrega información y no es una respuesta literal a lo que se pregunta como en el caso anterior.

**Documento similar 3:** En este caso la similitud baja considerablemente. Ya no se habla de soldadura y el tema no tiene nada que ver con el original, pero se menciona a una persona cuyo apellido es "Silver", por lo tanto se lo considera similar. Es posible que haya pocos textos sobre el tema tan específico de la soldadura con plata y el término "Silver" repetido tantas veces haya acercado los vectores. Esta es una limitación del modelo bag-of-words: no puede distinguir entre diferentes significados de la misma palabra.

**Documento similar 4:** Este documento vuelve a ser de la misma categoría y habla de soldadura, aunque no contiene plata. La similitud real es mucho mayor que la del documento 3.

**Documento similar 5:** Este documento tampoco tiene relación alguna con el original. También es una entrada en un post pero la única similitud es el nombre de quien firma, que es el mismo del texto original ("Andy").

In [208]:
analyze_document_similarity(sample_indices[2], X_test, newsgroups_test.data, y_test)

DOCUMENTO ORIGINAL (idx=1067) | Categoría: comp.sys.ibm.pc.hardware

Simple.  First, Andrew is correct, although I can see where there might be
some confusion.  It is indeed possible to have two cards *configured* to use
the same interrupt.  They can not *share* the interrupt in the sense that it
is not possible to have both cards active at the same time.

Here is an example.  For some time, I was short of "free interrupts."  I had a
tape controller (not a "floppy tape") that needed one of IRQ0-IRQ7.  (It's an
*old* tape drive.)  My solution was to use IRQ3 (also used for COM2, where my
modem is).  I did this because I reasoned I would never be using the modem and
the tape simultaneously.  When kermit runs, it installs its own interrupt
handler for IRQ3 and uses the serial port.  If the tape drive were to generate
an interrupt, kermit would not have a clue what to do with/for the tape
controller.  (And since the tape controller would not be serviced, it would
most likely "hang.")  Like

#### Análisis documento 3 (idx=1067)

Este documento es una respuesta en un post y habla sobre hardware interrupts (IRQ) y cómo se comparten o no los mismos.

**Documento similar 1:** También habla de IRQs. Se está discutiendo sobre el mismo tema, quizá con otros tipos de dispositivos. Se usan varios términos similares pero el score no es tan alto.

**Documento similar 2:** Este documento es también una entrada de un post y el autor es el mismo del texto original (Kenneth R. Ballou). Varios términos se repiten aunque no es tan específico como el original. El tema global es el mismo y la etiqueta también.

**Documento similar 3:** En este caso se pierde un poco la similaridad. Este texto habla de conexiones VCR que, si bien puede tener que ver con el hardware, no habla de IRQ y solo se considera similar por hablar de "tapes" pero en otro contexto.

**Documento similar 4:** Este texto tiene prácticamente el mismo score que el anterior pero tiene más relación. No explica sobre IRQ pero sí está consultando sobre el tema. Pareciera estar planteando un problema con un módem, pero habla de DOS e IRQs.

**Documento similar 5:** Este es el más alejado de todos. Aunque también varios términos se repiten, la relación es muy baja.

**Conclusión:** Para esta temática se encontraron textos mucho más similares. Al ser el texto más extenso y probablemente al haber más textos sobre estas temáticas, los textos no están tan alejados entre sí como en casos anteriores.

In [209]:
analyze_document_similarity(sample_indices[3], X_test, newsgroups_test.data, y_test)

DOCUMENTO ORIGINAL (idx=5637) | Categoría: misc.forsale
Title says it all, I'm after a Vectrex system. If you have one and want
to get rid of it let me know.

I can offer cash, or possible trades with Megadrive and SNES games.

Cheers
Marc

 ------------------------------------------------------------------------------
      **     **      *  ****** ***    *   |             On the net,
     ** *    **     *** **     ** *   *   |     no-one can hear you scream!
    **   *   **     *** ****   **  *  *   |------------------------------------
   **     *  **     *** **     **   * *   |  email   marc@comp.lancs.ac.uk
  **       * ******  *  ****** **    **   |      marc@computing.lancaster.ac.uk

────────────────────────────────────────────────────────────────────────────────
TOP 5 DOCUMENTOS MÁS SIMILARES:
────────────────────────────────────────────────────────────────────────────────

1. (idx=1464) [Score: 0.233] [✓] Categoría: misc.forsale
────────────────────────────────────────
If any

#### Análisis documento 4 (idx=5637)

Este documento pareciera ser un post en un blog de una persona en busca de un juego llamado Vectrex, por si alguien quisiera sacárselo de encima. Ofrece dinero en efectivo o intercambios por juegos.

**Documento similar 1:** Este primer documento es similar ya que también es un post en un blog y también busca a alguien que quiera sacarse de encima un juego y lo cambia por otros. Si bien no busca el mismo juego, propone un intercambio muy similar.

**Documento similar 2, 3 y 4:** Estos documentos no tienen mucha relación con el original. Sin embargo, los tres provienen de la misma persona y el saludo es igual al original ("Cheers, Marc"), pero la temática es diferente.

**Documento similar 5:** Este último documento no tiene mucha relación. Solo comparte una parte del texto ("If you have...") pero la temática no tiene ninguna relación más que hablar de software.

**Conclusión:** Este documento no tuvo tantas relaciones acertadas. Si bien en el primer caso había una relación, el resto solo se acercó por secuencias de texto similares sin contenido semántico similar.

In [210]:
analyze_document_similarity(sample_indices[4], X_test, newsgroups_test.data, y_test)

DOCUMENTO ORIGINAL (idx=4895) | Categoría: sci.med
=FLAME ON
=
=Reading through the posts about Kirlian (whatever spelling)
=photography I couldn't help but being slightly disgusted by the
=narrow-minded, "I know it all", "I don't believe what I can't see or
=measure" attitude of many people out there.

Where have you seen that attitude?

=I am neither a real believer, nor a disbeliever when it comes to
=so-called "paranormal" stuff; but as far as I'm concerned, it is just
=as likely as the existence of, for instance, a god, which seems to be
=quite accepted in our societies - without any scientific basis.

=I am convinced that it is a serious mistake to close your mind to
=something, ANYTHING, simply because it doesn't fit your current frame
=of reference. History shows that many great people, great scientists,
=were people who kept an open mind - and were ridiculed by sceptics.

Fine, jackass.  Suppose you point out even ONE aspect of Kirlian photography
that's not explained by a cor

#### Análisis documento 5 (idx=4895)

Este documento es una respuesta a un post en un blog. Se citan partes de un texto de un usuario y se responde. El tema de discusión es aparentemente sobre unas fotografías. Se discute sobre creencias paranormales versus una mirada más escéptica. Quien responde es quien fue acusado de no creer; se defiende al ser acusado de tener "cierta actitud" y finaliza insultando al otro usuario, justificando que las fotografías pueden ser explicadas bajo el fenómeno de "corona discharge".

**Documento similar 1:** Este documento es el post que es citado en el texto original, es el texto de la persona que lo acusa inicialmente. Hay una fuerte similitud principalmente por la cantidad de texto repetido y, dado que es una respuesta directa, tiene sentido que sea el más similar.

**Documento similar 2, 3, 4 y 5:** Todos estos textos son entradas del mismo hilo de conversación. La temática es la misma: una fotografía y su explicación esotérica o científica. Varios términos se repiten; de hecho, estos textos están escritos por la misma persona que escribió el texto original.

**Conclusión:** En este caso los scores son más altos que en los anteriores. Esto se explica ya que es el mismo hilo de conversación, mucho texto se repite y se cita.

En general, TF-IDF captura bien la similaridad léxica pero no semántica. Los documentos con texto citado/repetido tienen scores altos. Las firmas de autor y términos homónimos (Silver como apellido vs. metal) pueden generar falsos positivos.

### Desafío 1.2: Modelo de clasificación por prototipos
Según la consigna, se procede a obtener por cada documento en el conjunto de prueba el documento del conjunto de entrenamiento de mayor similitud coseno y asignarle así la misma etiqueta.

In [211]:
# Calcular similitudes para todo el conjunto de prueba respecto al conjunto de entrenamiento
similarities = cosine_similarity(X_test, X_train)
# Indices mas altos para cada documento de prueba
most_similar_indices = np.argmax(similarities, axis=1)

# Predicciones "zero-shot" basadas en la clase del documento más similar en el conjunto de entrenamiento
y_pred_zero_shot = y_train[most_similar_indices]

In [212]:
# comparar y_pred_zero_shot con y_test y obtener métricas
f1_score(y_test, y_pred_zero_shot, average='macro')

0.5049911553681621

Para la simplicidad del modelo zero-shot, el valor de la métrica obtenida está bastante razonable. Es diez veces mayor a una clasificación random dada la cantidad de categorías existentes, lo que indica que la similaridad coseno con TF-IDF captura información relevante sobre las categorías.

### Desafío 1.3: Entrenar Naive Bayes

Se realizan cuatro grid searches para buscar los parámetros y las combinaciones de modelo y vectorizacion que den los mejores resultados.
Se prueban las vectorizaciones TF-IDF y Count Vector y las variantes Multinomial y Complement de Naive Bayes.

#### 1. TD-IDF Vectorizer + MultinomialNB

In [213]:
tfidf_multinomial_pipeline = Pipeline([
    ('tfidf_vectorizer', TfidfVectorizer(sublinear_tf=True, stop_words='english')),
    ('multinomialNB', MultinomialNB())
])

In [214]:
parameters = {
    'multinomialNB__alpha': [0.05, 0.1, 0.2, 0.5], 
    'tfidf_vectorizer__max_features': [10000, 15000, 30000, 50000], 
    'tfidf_vectorizer__max_df': [0.5, 0.75, 1.0], 
    'tfidf_vectorizer__min_df': [1, 5, 10]
}
clf = GridSearchCV(tfidf_multinomial_pipeline, parameters, scoring='f1_macro', cv=5, n_jobs=-1)
clf.fit(newsgroups_train.data, newsgroups_train.target)
print("Best parameters set found on development set:")
print(clf.best_params_)

y_pred = clf.predict(newsgroups_test.data)
f1_macro = f1_score(y_test, y_pred, average='macro')
print(f"F1 macro score: {f1_macro}")

Best parameters set found on development set:
{'multinomialNB__alpha': 0.05, 'tfidf_vectorizer__max_df': 0.5, 'tfidf_vectorizer__max_features': 50000, 'tfidf_vectorizer__min_df': 1}
F1 macro score: 0.6790030944545424


#### 2. TD-IDF Vectorizer + ComplementNB

In [215]:
tfidf_complementNB_pipeline = Pipeline([
    ('tfidf_vectorizer', TfidfVectorizer(sublinear_tf=True, stop_words='english')),
    ('complementNB', ComplementNB())
])

In [216]:
parameters = {
    'complementNB__alpha': [0.1, 0.3, 0.5, 1], 
    'tfidf_vectorizer__max_features': [15000, 30000, 50000, 100000, None], 
    'tfidf_vectorizer__max_df': [0.5, 0.75, 0.8, 1.0], 
    'tfidf_vectorizer__min_df': [1, 5, 10]
}
clf_complement = GridSearchCV(tfidf_complementNB_pipeline, parameters, scoring='f1_macro', cv=5, n_jobs=-1)
clf_complement.fit(newsgroups_train.data, newsgroups_train.target)
print("Best parameters set found on development set:")
print(clf_complement.best_params_)

y_pred = clf_complement.predict(newsgroups_test.data)
f1_macro = f1_score(y_test, y_pred, average='macro')
print(f"F1 macro score: {f1_macro}")

Best parameters set found on development set:
{'complementNB__alpha': 0.3, 'tfidf_vectorizer__max_df': 0.5, 'tfidf_vectorizer__max_features': 100000, 'tfidf_vectorizer__min_df': 1}
F1 macro score: 0.697776135220974


#### 3. Count Vectorizer + MultinomialNB

In [217]:
count_multinomial_pipeline = Pipeline([
    ('count_vectorizer', CountVectorizer(stop_words='english')),
    ('multinomialNB', MultinomialNB())
])

In [218]:
parameters = {
    'multinomialNB__alpha': [0.05, 0.1, 0.2, 0.5], 
    'count_vectorizer__max_features': [10000, 15000, 30000, 50000], 
    'count_vectorizer__max_df': [0.5, 0.75, 1.0], 
    'count_vectorizer__min_df': [1, 5, 10]
}
clf = GridSearchCV(count_multinomial_pipeline, parameters, scoring='f1_macro', cv=5, n_jobs=-1)
clf.fit(newsgroups_train.data, newsgroups_train.target)
print("Best parameters set found on development set:")
print(clf.best_params_)

y_pred = clf.predict(newsgroups_test.data)
f1_macro = f1_score(y_test, y_pred, average='macro')
print(f"F1 macro score: {f1_macro}")

Best parameters set found on development set:
{'count_vectorizer__max_df': 0.5, 'count_vectorizer__max_features': 50000, 'count_vectorizer__min_df': 1, 'multinomialNB__alpha': 0.05}
F1 macro score: 0.6275847141923997


#### 4. Count Vectorizer + ComplementNB

In [219]:
count_complement_pipeline = Pipeline([
    ('count_vectorizer', CountVectorizer(stop_words='english')),
    ('complementNB', ComplementNB())
])

In [220]:
parameters = {
    'complementNB__alpha': [0.1, 0.3, 0.5, 1], 
    'count_vectorizer__max_features': [15000, 30000, 50000, 100000], 
    'count_vectorizer__max_df': [0.5, 0.75, 0.8, 1.0], 
    'count_vectorizer__min_df': [1, 5, 10]
}
clf_complement = GridSearchCV(count_complement_pipeline, parameters, scoring='f1_macro', cv=5, n_jobs=-1)
clf_complement.fit(newsgroups_train.data, newsgroups_train.target)
print("Best parameters set found on development set:")
print(clf_complement.best_params_)

y_pred = clf_complement.predict(newsgroups_test.data)
f1_macro = f1_score(y_test, y_pred, average='macro')
print(f"F1 macro score: {f1_macro}")

Best parameters set found on development set:
{'complementNB__alpha': 0.3, 'count_vectorizer__max_df': 0.5, 'count_vectorizer__max_features': 100000, 'count_vectorizer__min_df': 1}
F1 macro score: 0.6469921609076227


#### Resultados

| Vectorizer | Modelo | F1 Macro |
|------------|--------|----------|
| TF-IDF | MultinomialNB | 0.679 |
| TF-IDF | ComplementNB | **0.698** |
| Count | MultinomialNB | 0.628 |
| Count | ComplementNB | 0.647 |

El mejor modelo es el TF-IDF + ComplementNB con un F1 Macro de 0.698 que es un valor bastante más alto que el obtenido con el modelo zero-shot del punto anterior. 

El vectorizador TF-IDF da mejores resultados que en el CountVector, esto es ya que TF-IDF penaliza a las palabras que se repiten y le da más peso a las palabras menos frecuentes o distintivas, mientras que CountVector solo cuenta las frecuencias de las palabras.

ComplementNB supera a MultinomialNB en ambos casos. Mientras que Multinomial aprende de las palabras más comunes dentro de la clase, Complement aprende qué palabras aparecen en documentos que no pertenecen a cada clase. Además ComplementNB suele funcionar mejor cuando hay clases desbalanceadas o cuando algunas clases tienen documentos más largos que otras.

Todos los modelos prefieren `max_df=0.5` (eliminar palabras muy frecuentes) y `min_df=1` (no eliminar palabras raras)

### Desafío 4: Transponer la matriz de documento-termino y obtener similitud entre palabras

In [221]:
term_doc_matrix = X_train.T

words = ['chicago', 'space', 'car', 'religion', 'health']

for word in words:
    word_idx = tfidfvect.vocabulary_[word]
    word_vector = term_doc_matrix[word_idx]

    similarities = cosine_similarity(word_vector, term_doc_matrix)[0]

    top_n = np.argpartition(similarities, -6)[-6:]  # 6 porque incluye a sí misma
    top_n_sorted = top_n[np.argsort(similarities[top_n])[::-1]]

    print(f"\n'{word}' - palabras similares:")
    for idx in top_n_sorted:
        if idx != word_idx:  # excluir la palabra misma
            print(f"  {idx2word[idx]}: {similarities[idx]:.3f}")



'chicago' - palabras similares:
  toronto: 0.257
  detroit: 0.254
  pittsburgh: 0.223
  calgary: 0.220
  angeles: 0.208

'space' - palabras similares:
  nasa: 0.330
  seds: 0.297
  shuttle: 0.293
  enfant: 0.280
  seti: 0.246

'car' - palabras similares:
  cars: 0.180
  criterium: 0.177
  civic: 0.175
  owner: 0.169
  dealer: 0.168

'religion' - palabras similares:
  religious: 0.245
  religions: 0.212
  categorized: 0.204
  purpsoe: 0.201
  crusades: 0.199

'health' - palabras similares:
  ohip: 0.330
  provincial: 0.300
  care: 0.282
  untouchable: 0.280
  fiscally: 0.280


#### Análisis de palabras similares

**Chicago:** Todas las palabras similares son ciudades (Toronto, Detroit, Pittsburgh, Calgary, Los Angeles). Esto tiene sentido ya que estas ciudades probablemente aparecen en contextos similares (noticias, deportes, etc.). Sin embargo, los scores obtenidos no son tan altos.

**Space:** Se obtienen NASA, SEDS, shuttle y SETI, que son organizaciones y términos relacionados con el estudio del espacio, por lo que la relación entre estas palabras es alta. La palabra "enfant" no tiene relación aparente; sin embargo revisando el corpus se confirma que viene de la dirección de la oficina de la NASA en Washington que se encuentra en la dirección "955 L'Enfant Plaza S.W."

**Car:** Todas las palabras tienen relación excepto "criterium". Dentro de las palabras relacionadas encontramos: "cars" (básicamente la misma palabra), "civic" (nombre de un modelo de auto Honda), "owner" y "dealer" que forman conceptos compuestos como "car owner" y "car dealer", por lo que las palabras se relacionan entre ellas. Buscando en el corpus criterium viene de safe criterium que es un concepto relevante en el momento de la compra de un auto.

**Religion:** Las primeras dos palabras comparten la misma raíz: "religious" y "religions". "Crusades" (cruzadas) fueron guerras religiosas, por lo que la relación es clara. "Purpsoe" parece ser un error tipográfico de "purpose"; puede haber una relación entre el propósito y la religión. Finalmente, "categorized" no tendría mucho sentido fuera de algún contexto específico.

**Health:** "OHIP" es el sistema de salud público de Ontario; deben haber muchos textos que posean esta sigla. "Provincial" puede ser un caso similar, relacionado con el contexto canadiense de salud pública. "Care" está claramente relacionado por el concepto de "health care". "Untouchable" y "fiscally" no tienen una relación clara sin más contexto.

**Conclusión:** Se demuestra cómo el corpus dado no es tan amplio y las palabras similares que devuelve están atadas al contexto específico de los documentos, por lo que no siempre son fáciles de interpretar. A diferencia de embeddings semánticos (como Word2Vec), TF-IDF captura co-ocurrencia en documentos, no significado semántico.

In [228]:
word = "enfant"
docs_with_word = [i for i, doc in enumerate(newsgroups_train.data) if word.lower() in doc.lower()]

print(f'Found {len(docs_with_word)} documents containing "{word}"\n')

for idx in docs_with_word[:3]:  # Show first 3
    print(f"Document {idx}:")
    print(newsgroups_train.data[idx])
    print("-" * 80)

Found 4 documents containing "enfant"

Document 2800:
Archive-name: space/net
Last-modified: $Date: 93/04/01 14:39:15 $

NETWORK RESOURCES

OVERVIEW

    You may be reading this document on any one of an amazing variety of
    computers, so much of the material below may not apply to you. In
    general, however, systems connected to 'the net' fall in one of three
    categories: Internet, Usenet, or BITNET. Electronic mail may be sent
    between these networks, and other resources available on one of these
    networks are sometimes accessible from other networks by email sent to
    special 'servers'.

    The space and astronomy discussion groups actually are composed of
    several mechanisms with (mostly) transparent connections between them.

    One mechanism is the mailing list, in which mail is sent to a central
    distribution point which relays it to all recipients of the list. In
    addition to the general lists for space (called SPACE Digest for
    Internet users, and 